In [1]:
import numpy as np
import numpy.lib.recfunctions as recfun
from pathlib import Path
from plyfile import PlyData, PlyElement
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import open3d as o3d
import pandas as pd

from matplotlib import cm
import copy

import random
import os
import gc
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import MulticlassF1Score, MulticlassPrecision, MulticlassRecall, MulticlassJaccardIndex, BinaryAccuracy, BinaryMatthewsCorrCoef, MulticlassConfusionMatrix

from sklearn.model_selection import KFold

import torchsummary
import time

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def set_seed(seed=42):
    # Python & OS
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch CPU
    torch.manual_seed(seed)
    
    # PyTorch GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
    print(f"Global seed fixed on {seed}")

In [3]:
set_seed()

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

Global seed fixed on 42


In [4]:
def load_full_dataset_ssm_lb(dataset='Train'):
    # 1) Path Definition
    # base directory for the chosen dataset
    base_path = Path('../data/Challenge-ABC') / dataset
    if not base_path.exists(): # verification
        print(f"Error: Dataset folder {base_path} not found, check 'dataset' argument.")
        return None
    ssm_dir = base_path / 'SSM_Challenge-ABC'
    lb_dir = base_path / 'lb'

    X_list = []
    y_list = []

    ssm_files = list(ssm_dir.glob('*.ssm'))
    total_files = len(ssm_files)

    for i, ssm_file in enumerate(ssm_files, 1):
        file_id = ssm_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        labels = np.loadtxt(lb_file, dtype='int64')

        if len(labels) == len(features):
            X_list.append(features)
            y_list.append(labels)
        else:
            print(f"Error: Dimension error for {file_id} -> ignored.")

        #if i % 20 == 0:
            #print(f"Loading : {i}/{total_files} files...")

    print(f"Total of {total_files} .ssm and .lb files has been loaded successfully.")
    X = np.vstack(X_list)
    y = np.concatenate(y_list)    
    
    return X, y

In [5]:
def load_full_dataset_ply_lb(dataset='Train'):
    base_path = Path('../data/Challenge-ABC') / dataset
    if not base_path.exists():
        print(f"Error: Dataset folder {base_path} not found.")
        return None, None, None

    ply_dir = base_path / 'ply'
    lb_dir = base_path / 'lb'

    ply_files = list(ply_dir.glob('*.ply'))
    total_files = len(ply_files)

    points_list = []
    labels_list = []
    file_ids = []

    for i, ply_file in enumerate(ply_files, 1):
        file_id = ply_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # Check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        # Load labels
        labels = np.loadtxt(lb_file, dtype='int64')
        
        # Load point cloud using Open3D
        pcd = o3d.io.read_point_cloud(str(ply_file))
        points = np.asarray(pcd.points)

        # Check dimension consistency
        if len(points) != len(labels):
            print(f"Error: Dimension error with {file_id} : {len(points)} points vs {len(labels)} labels -> ignored.")
            continue

        # Append to lists to maintain object separation
        points_list.append(points)
        labels_list.append(labels)
        file_ids.append(file_id)

        # Progress tracking
        #if i % 20 == 0:
           #print(f"Loading : {i}/{total_files} files...")

    print(f"Total of {total_files} .ply and .lb files has been loaded successfully.")

    return points_list, labels_list, file_ids

In [6]:
def downsample_non_edges(points, labels, method, display=False, **kwargs):
    # 1. Initialize structure and extract global masks
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    global_edge_idx = np.where(labels == 1)[0]
    global_non_edge_idx = np.where(labels != 1)[0] # Security; using != 1 to catch any unclassified points
    
    non_edge_pcd = pcd.select_by_index(global_non_edge_idx)
    
    # 2. Routing logic for the selected algorithm
    method = method.lower()
    if method == 'voxel':
        if 'resolution_percentage' not in kwargs:
            raise ValueError("Method 'voxel' requires 'resolution_percentage'")
        
        # Calculate bounds on the full point cloud to maintain consistent relative scale
        min_bound = points.min(axis=0)
        max_bound = points.max(axis=0)
        max_dim = np.max(max_bound - min_bound)
        dynamic_voxel_size = max_dim * kwargs['resolution_percentage']
        
        _, _, trace = non_edge_pcd.voxel_down_sample_and_trace(
            voxel_size=dynamic_voxel_size, min_bound=min_bound, max_bound=max_bound
        )
        rel_idx = np.array([v[np.random.randint(0, len(v))] for v in trace])
        selected_non_edge_global_idx = global_non_edge_idx[np.sort(rel_idx)]
        
    elif method == 'fps':
        if 'retention_rate' in kwargs:
            target_pts = max(1, int(len(global_non_edge_idx) * kwargs['retention_rate']))
        elif 'num_points' in kwargs:
            target_pts = min(kwargs['num_points'], len(global_non_edge_idx))
        else:
            raise ValueError("Method 'fps' requires 'retention_rate' or 'num_points'")
        
        fps_pcd = non_edge_pcd.farthest_point_down_sample(target_pts)
        tree = o3d.geometry.KDTreeFlann(non_edge_pcd)
        rel_idx = np.zeros(target_pts, dtype=int)
        
        for i, pt in enumerate(fps_pcd.points):
            rel_idx[i] = tree.search_knn_vector_3d(pt, 1)[1][0]
            
        selected_non_edge_global_idx = global_non_edge_idx[rel_idx]

    elif method == 'poisson':
        if 'radius' not in kwargs:
            raise ValueError("Method 'poisson' requires 'radius'")
        tree = o3d.geometry.KDTreeFlann(non_edge_pcd)
        pts = np.asarray(non_edge_pcd.points)
        num_pts = len(pts)
        
        active_mask = np.ones(num_pts, dtype=bool)
        shuffled_idx = np.random.permutation(num_pts)
        rel_idx = []
        
        for idx in shuffled_idx:
            if not active_mask[idx]: continue
            rel_idx.append(idx)
            _, neighbors_idx, _ = tree.search_radius_vector_3d(pts[idx], kwargs['radius'])
            active_mask[np.asarray(neighbors_idx)] = False
            
        selected_non_edge_global_idx = global_non_edge_idx[rel_idx]

    elif method == 'random':
        if 'retention_rate' in kwargs:
            target_pts = max(1, int(len(global_non_edge_idx) * kwargs['retention_rate']))
        elif 'num_points' in kwargs:
            target_pts = min(kwargs['num_points'], len(global_non_edge_idx))
        else:
            raise ValueError("Method 'random' requires 'retention_rate' or 'num_points'")
        
        selected_non_edge_global_idx = np.random.choice(global_non_edge_idx, size=target_pts, replace=False)
        selected_non_edge_global_idx = np.sort(selected_non_edge_global_idx)

    elif method == 'uniform':
        if 'k_step' not in kwargs:
            raise ValueError("Method 'uniform' requires 'k_step'")
        selected_non_edge_global_idx = global_non_edge_idx[::kwargs['k_step']]
    else:
        raise ValueError(f"Unknown method '{method}'")

    # 3. Final Reintegration
    final_global_idx = np.concatenate([global_edge_idx, selected_non_edge_global_idx])
    final_global_idx = np.sort(final_global_idx)

    # 4. Diagnostics and Visualization
    if display:
        print(f"\n--- Downsampling Report ({method.upper()}) ---")
        print(f"Original Cloud Size : {len(points)}")
        print(f"Edge Points Kept    : {len(global_edge_idx)}")
        print(f"Non-Edge Downsampled: {len(selected_non_edge_global_idx)}")
        print(f"Final Cloud Size    : {len(final_global_idx)}")
        
        if method == 'voxel':
            print(f"Computed Voxel Size : {dynamic_voxel_size:.4f}")
        
        inter_pcd = pcd.select_by_index(selected_non_edge_global_idx)
        res_pcd = pcd.select_by_index(final_global_idx)
        
        res_pcd.paint_uniform_color([0, 0, 1])
        inter_pcd.paint_uniform_color([0, 0, 1])
        edge_colors = np.zeros((len(final_global_idx), 3))
        edge_colors[:] = [0, 0, 1]
        
        edge_mask_in_final = np.isin(final_global_idx, global_edge_idx)
        edge_colors[edge_mask_in_final] = [1, 0, 0] 
        res_pcd.colors = o3d.utility.Vector3dVector(edge_colors)
        

        o3d.visualization.draw_plotly([inter_pcd])
        o3d.visualization.draw_plotly([res_pcd])

    return final_global_idx

In [7]:
def build_downsampled_features(points_list, labels_list, file_ids, method, dataset='Train', **kwargs):
    ssm_dir = Path('../data/Challenge-ABC') / dataset / 'SSM_Challenge-ABC'
    
    X_list = []
    y_list = []
    
    total_files = len(file_ids)
    print(f"Starting feature extraction using '{method}' downsampling...")

    for i in range(total_files):
        file_id = file_ids[i]
        points = points_list[i]
        labels = labels_list[i]
        
        # Compute the global indices to keep for this specific model
        final_idx = downsample_non_edges(points, labels, method=method, display=False, **kwargs)
        
        # Load the corresponding .ssm file
        ssm_file = ssm_dir / f"{file_id}.ssm"
        if not ssm_file.exists():
            print(f"Error: {ssm_file.name} not found -> ignored.")
            continue
        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        
        # Dimension consistency check
        if len(features) != len(points):
            print(f"Error: Dimension mismatch in {file_id}.ssm ({len(features)} features vs {len(points)} points) -> ignored.")
            continue
            
        # Slice the arrays using the downsampled indices
        filtered_features = features[final_idx]
        filtered_labels = labels[final_idx]
        
        X_list.append(filtered_features)
        y_list.append(filtered_labels)
        
        #if (i + 1) % 10 == 0:
            #print(f"Processed {i + 1}/{total_files} feature files...")
            
    print(f"Feature extraction complete. Processed {len(X_list)} valid files.\n")
    
    return X_list, y_list

In [8]:
class MLP(nn.Module):
    def __init__(self, input_dim=320, num_hidden_1=64, num_hidden_2=32):
        super(MLP, self).__init__()
        self.layer_1 = torch.nn.Linear(input_dim, num_hidden_1)
        self.layer_2 = torch.nn.Linear(num_hidden_1, num_hidden_2)
        self.layer_3 = torch.nn.Linear(num_hidden_2, 2)
        self.num_hidden_1 = num_hidden_1
        self.num_hidden_2 = num_hidden_2

    def forward(self, x):
        # Flatten the input from [batch_size, 20, 16] to [batch_size, 320]
        x = x.view(x.size(0), -1)

        # 1st layer
        out = self.layer_1(x)
        #out = torch.tanh(out) 
        out = torch.relu(out)
        
        # 2nd layer
        out = self.layer_2(out)
        out = torch.relu(out)

        # 3rd/output layer
        out = self.layer_3(out)
        return out

In [9]:
# Train Function
def train_mlp_model(X_train, y_train, X_val, y_val, nb_epochs=50, batch_size=1024, lr=0.01, device='cpu', display_metrics=False, seed=42, patience=10):
    set_seed(seed)

    input_dim = X_train.size(1) 
    
    start_time = time.time()
    print(f"Training on device: {device} (Seed: {seed}) | Features in input: {input_dim}")
    model = MLP(input_dim=input_dim).to(device)
    X_train_gpu = X_train.to(device)
    y_train_gpu = y_train.to(device)
    X_val_gpu = X_val.to(device)
    y_val_gpu = y_val.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    #optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    # --- Early Stopping and Best Model SEtup --
    best_val_loss = float('inf')
    nb_epochs_no_improvement = 0
    best_epoch = 1
    best_model_weights = copy.deepcopy(model.state_dict) 

    # --- Metrics ---
    # Dictionnary to store the complete history
    history = {
        'train_loss_epoch': [], 'train_loss_step': [], 'val_loss': [], 
        'val_acc': [], 'val_mcc': [],
        'val_precision_0': [], 'val_precision_1': [], 
        'val_recall_0': [], 'val_recall_1': [], 
        'val_f1_0': [], 'val_f1_1': [], 
        'val_iou_0': [], 'val_iou_1': [],
        'val_tp':[], 'val_tn':[], 'val_fp':[], 'val_fn':[]
    }


    # use Multiclass with average='none' to get a tensor with [score_class_0, score_class_1]
    f1_metric = MulticlassF1Score(num_classes=2, average='none').to(device)
    precision_metric = MulticlassPrecision(num_classes=2, average='none').to(device)
    recall_metric = MulticlassRecall(num_classes=2, average='none').to(device)
    iou_metric = MulticlassJaccardIndex(num_classes=2, average='none').to(device)
    mcc_metric = BinaryMatthewsCorrCoef().to(device)
    acc_metric = BinaryAccuracy().to(device)
    conf_matrix_metric = MulticlassConfusionMatrix(num_classes=2).to(device)

    
    print("Start of training...")
    
    for epoch in range(1, nb_epochs + 1):

        # --- Training Phase ---
        model.train()
        running_train_loss = 0.0
        
        # Create random permutation of indexes to shuffle the dataset at each epoch withour copying the whole dataset 
        idx = torch.randperm(X_train_gpu.size(0), device=device)

        # Loop over the dataset but instead of copying and shuffling the whole dataset, we work on the indexes 
        # so we slice only the indexes and pick the data directly from the original tensor based on the shuffled indexes
        for i in range(0, X_train_gpu.size(0), batch_size):
            # extract only the indexes for the current batch
            batch_idx = idx[i : i + batch_size]

            # pick the batch data from the data tensor 
            inputs = X_train_gpu[batch_idx]
            labels = y_train_gpu[batch_idx]

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            # added the loss at each step to the history for more detailed overview
            history['train_loss_step'].append(loss.item())
            running_train_loss += loss.item() * inputs.size(0)   
            
        epoch_train_loss = running_train_loss / X_train_gpu.size(0)
        history['train_loss_epoch'].append(epoch_train_loss)

        # --- Validation Phase ---
        model.eval()
        running_val_loss = 0.0

        f1_metric.reset()
        precision_metric.reset()
        recall_metric.reset()
        mcc_metric.reset()
        iou_metric.reset()
        acc_metric.reset()
        conf_matrix_metric.reset()

        with torch.no_grad():
            # same idea but without shuffling and no need to compute gradients
            for i in range(0, X_val_gpu.size(0), batch_size):
                inputs = X_val_gpu[i : i + batch_size]
                labels = y_val_gpu[i : i + batch_size]
      
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs, 1)

                f1_metric.update(predicted, labels)
                precision_metric.update(predicted, labels)
                recall_metric.update(predicted, labels)
                mcc_metric.update(predicted, labels)
                iou_metric.update(predicted, labels)
                acc_metric.update(predicted, labels)
                conf_matrix_metric.update(predicted, labels)
                
        epoch_val_loss = running_val_loss / X_val_gpu.size(0)
        history['val_loss'].append(epoch_val_loss)
        
        
        # --- Compute Evaluation Metrics ---
        # now compute() returns a tensor of size 2 : [score_class_0, score_class_1]
        val_prec = precision_metric.compute()
        val_rec = recall_metric.compute()
        val_f1 = f1_metric.compute()
        val_iou = iou_metric.compute()
        conf_mat = conf_matrix_metric.compute()
        
        history['val_precision_0'].append(val_prec[0].item())
        history['val_precision_1'].append(val_prec[1].item())
        history['val_recall_0'].append(val_rec[0].item())
        history['val_recall_1'].append(val_rec[1].item())
        history['val_f1_0'].append(val_f1[0].item())
        history['val_f1_1'].append(val_f1[1].item())
        history['val_iou_0'].append(val_iou[0].item())
        history['val_iou_1'].append(val_iou[1].item())

        # global metrics computed as before
        val_mcc = mcc_metric.compute().item()
        val_acc = acc_metric.compute().item() * 100.0
        
        history['val_mcc'].append(val_mcc)
        history['val_acc'].append(val_acc)

        history['val_tn'].append(conf_mat[0, 0].item())
        history['val_fp'].append(conf_mat[0, 1].item())
        history['val_fn'].append(conf_mat[1, 0].item())
        history['val_tp'].append(conf_mat[1, 1].item())

        if display_metrics:
            print(f"Epoch [{epoch}/{nb_epochs}] | T.Loss: {epoch_train_loss:.4f} | V.Loss: {epoch_val_loss:.4f} | F1 (Class 1): {val_f1[1].item():.4f} | MCC: {val_mcc:.4f}")

        # --- Early Stopping Logic ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch
            best_model_weights = copy.deepcopy(model.state_dict())
            nb_epochs_no_improvement = 0
        else:
            nb_epochs_no_improvement += 1
        
        if nb_epochs_no_improvement >= patience:
            print(f"\nEarly Stopping triggered at epoch {epoch}. Returning best weigths to now.")
            break

    history['best_epoch'] = best_epoch
    
    # --- Best Model ---
    print(f"Getting best model weights with val_loss = {best_val_loss:.4f}")
    model.load_state_dict(best_model_weights)

    # Free VRAM data 
    del X_train_gpu, y_train_gpu, X_val_gpu, y_val_gpu, f1_metric, precision_metric, recall_metric, mcc_metric, iou_metric, acc_metric, conf_matrix_metric
    torch.cuda.empty_cache()
    
    end_time = time.time()
    print(f"Training took {(end_time - start_time) / 60:.2f} minutes.")
    
    return history, model

In [10]:
def plot_metrics(metrics_history):
    fig, axs = plt.subplots(5, 2, figsize=(20, 25), gridspec_kw={'height_ratios': [1, 1, 1, 1, 1.5]})

    # Train Loss per step
    axs[0, 0].set_title("Train Loss (per step)")
    axs[0, 0].plot(metrics_history['train_loss_step'], color='blue', alpha=0.75, linewidth=0.75)
    axs[0, 0].set_xlabel("Iterations (Batches)")
    axs[0, 0].set_ylabel("Loss")
    axs[0, 0].grid(True, alpha=0.5)

    # Loss per Epoch Train and Val
    axs[0, 1].set_title("Loss (per Epoch)")
    axs[0, 1].plot(metrics_history['train_loss_epoch'], label="Train Loss", marker='o', color='blue')
    axs[0, 1].plot(metrics_history['val_loss'], label="Validation Loss", marker='o', color='red')
    axs[0, 1].set_xlabel("Epoch")
    axs[0, 1].set_ylabel("Loss")
    axs[0, 1].legend()
    axs[0, 1].grid(True, alpha=0.5)

    # Precision and Recall Class 1
    axs[1, 0].set_title("Precision & Recall (Class 1 - Validation)")
    axs[1, 0].plot(metrics_history['val_precision_1'], label='Precision', color='red')
    axs[1, 0].plot(metrics_history['val_recall_1'], label='Recall', color='green')
    axs[1, 0].set_xlabel("Epoch")
    axs[1, 0].set_ylabel("Score")
    axs[1, 0].legend()
    axs[1, 0].grid(True, alpha=0.5)

    # Precision and Recall Class 0
    axs[1, 1].set_title("Precision & Recall (Class 0 - Validation)")
    axs[1, 1].plot(metrics_history['val_precision_0'], label='Precision', color='red')
    axs[1, 1].plot(metrics_history['val_recall_0'], label='Recall', color='green')
    axs[1, 1].set_xlabel("Epoch")
    axs[1, 1].set_ylabel("Score")
    axs[1, 1].legend()
    axs[1, 1].grid(True, alpha=0.5)

    # F1-Score
    axs[2, 0].set_title("F1-Score per Class (Validation)")
    axs[2, 0].plot(metrics_history['val_f1_0'], label="F1 - Class 0", linestyle='--', color='blue')
    axs[2, 0].plot(metrics_history['val_f1_1'], label="F1 - Class 1", linewidth=2, color='red')
    axs[2, 0].set_xlabel("Epoch")
    axs[2, 0].set_ylabel("Score")
    axs[2, 0].legend()
    axs[2, 0].grid(True, alpha=0.5)

    # Global Metrics (MCC) & IoU
    axs[2, 1].set_title("MCC (Global) & IoU (per Class)")
    axs[2, 1].plot(metrics_history['val_mcc'], label="MCC (Global)", color='brown', marker='s')
    axs[2, 1].plot(metrics_history['val_iou_0'], label="IoU - Class 0", linestyle='--', color='blue')
    axs[2, 1].plot(metrics_history['val_iou_1'], label="IoU - Class 1", linewidth=2, color='red')
    axs[2, 1].set_xlabel("Epoch")
    axs[2, 1].set_ylabel("Score")
    axs[2, 1].legend()
    axs[2, 1].grid(True, alpha=0.5)

    # Class 1 Prediction tracking TP and FN 
    axs[3, 0].set_title("Edge Tracking: TP and FN")
    axs[3, 0].plot(metrics_history['val_tp'], label="TP = Correct Edges", color='green', marker='^')
    axs[3, 0].plot(metrics_history['val_fn'], label="FN = Missed Edges", color='red', marker='v')
    axs[3, 0].set_xlabel("Epoch")
    axs[3, 0].set_ylabel("Count")
    axs[3, 0].set_yscale('log')
    axs[3, 0].legend()
    axs[3, 0].grid(True, alpha=0.5)

    # Class 0 Prediction tracking TN and FP 
    axs[3, 1].set_title("Non-Edge Tracking: TN and FP")
    axs[3, 1].plot(metrics_history['val_tn'], label="TN = Correct Non-Edges", color='green', marker='^')
    axs[3, 1].plot(metrics_history['val_fp'], label="FP = Fake Edges", color='red', marker='v')
    axs[3, 1].set_xlabel("Epoch")
    axs[3, 1].set_ylabel("Count")
    axs[3, 1].set_yscale('log')
    axs[3, 1].legend()
    axs[3, 1].grid(True, alpha=0.5)

    # Best Epoch Data Extraction
    # Confusion Matrix of the best epoch
    best_epoch = metrics_history['best_epoch']
    best_idx = best_epoch - 1
    best_cm = np.array([
        [metrics_history['val_tn'][best_idx], metrics_history['val_fp'][best_idx]],
        [metrics_history['val_fn'][best_idx], metrics_history['val_tp'][best_idx]]
    ])

    hex_colors = ['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4']
    color_map = ListedColormap(hex_colors) 
    color_indices = np.array([
        [0, 1], 
        [2, 3]  
    ])
    axs[4, 0].matshow(color_indices, cmap=color_map)
    
    axs[4, 0].set_title(f"Confusion Matrix - Best Epoch: {best_epoch}")
    axs[4, 0].set_xlabel("Predicted Label")
    axs[4, 0].set_ylabel("True Label")
    axs[4, 0].set_xticks([0, 1])
    axs[4, 0].set_yticks([0, 1])
    axs[4, 0].set_xticklabels(['Class 0', 'Class 1'])
    axs[4, 0].set_yticklabels(['Class 0', 'Class 1'])

    for (i, j), val in np.ndenumerate(best_cm):
        axs[4, 0].text(j, i, f"{int(val)}", ha='center', va='center', color='white', fontsize=16, fontweight='bold')

    # custom legend elements mapping the colors to the labels
    legend_elements = [
        Patch(facecolor=hex_colors[0], edgecolor='gray', label='TN (Correct Non-Edges)'),
        Patch(facecolor=hex_colors[1], edgecolor='gray', label='FP (Fake Edges)'),
        Patch(facecolor=hex_colors[2], edgecolor='gray', label='FN (Missed Edges)'),
        Patch(facecolor=hex_colors[3], edgecolor='gray', label='TP (Correct Edges)')
    ]

    axs[4, 0].legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), 
                     fontsize=10, frameon=True, edgecolor='gray')


    # Summary Text Box
    axs[4, 1].axis('off') # Hide axes
    summary_text = (
        f"BEST MODEL METRICS (Epoch {best_epoch})\n\n"
        f"Validation Loss :  {metrics_history['val_loss'][best_idx]:.4f}\n"
        f"Accuracy        :  {metrics_history['val_acc'][best_idx]:.2f}%\n"
        f"MCC (Global)    :  {metrics_history['val_mcc'][best_idx]:.4f}\n\n"
        f"-- CLASS 1 (EDGES) --\n"
        f"F1-Score        :  {metrics_history['val_f1_1'][best_idx]:.4f}\n"
        f"Precision       :  {metrics_history['val_precision_1'][best_idx]:.4f}\n"
        f"Recall          :  {metrics_history['val_recall_1'][best_idx]:.4f}\n\n"
        f"-- CLASS 0 (NON-EDGES) --\n"
        f"F1-Score        :  {metrics_history['val_f1_0'][best_idx]:.4f}\n"
        f"Precision       :  {metrics_history['val_precision_0'][best_idx]:.4f}\n"
        f"Recall          :  {metrics_history['val_recall_0'][best_idx]:.4f}\n"
    )

    axs[4, 1].text(0.5, 0.5, summary_text, fontsize=14, va='center', ha='center', family='monospace', 
                bbox=dict(facecolor="#f0f0f0", edgecolor='gray', alpha=0.8, boxstyle='round,pad=1.5'))


    plt.tight_layout()
    plt.show()

In [11]:
def normalize_data_inplace(X_tensor, mean=None, std=None, opt='flat', num_desc=20, num_scales=16):
    # start by making sure to have a shape of N, 320 
    X_tensor = X_tensor.view(X_tensor.size(0), -1)

    if opt == 'flat':
        if mean is None or std is None:
            mean = X_tensor.mean(dim=0, keepdim=True)
            std = X_tensor.std(dim=0, keepdim=True) + 1e-8 # avoid zero division
        X_tensor.sub_(mean).div_(std) # in-place operations

    elif opt == 'desc':
        computed_means, computed_stds = [], []
        for i in range(num_desc):
            start_idx = i * num_scales
            end_idx = (i + 1) * num_scales
            chunk = X_tensor[:, start_idx:end_idx] # a chunk = a descriptor

            if mean is None or std is None:
                chunk_mean = chunk.mean().item()
                chunk_std = chunk.std().item() + 1e-8
                computed_means.append(chunk_mean)
                computed_stds.append(chunk_std)
            else:
                chunk_mean = mean[0, i].item()
                chunk_std = std[0, i].item()

            chunk.sub_(chunk_mean).div_(chunk_std)

        if mean is None or std is None:
            mean = torch.tensor(computed_means, dtype=torch.float32).view(1, num_desc)
            std = torch.tensor(computed_stds, dtype=torch.float32).view(1, num_desc)

    else:
        raise ValueError("opt must be 'desc' or 'flat'")

    return X_tensor, mean, std

In [12]:
def kfold_cross_val_train(train_pts, train_lb, train_ids, k=5 , ds_method='voxel', norm_opt='flat', device='cuda', seed=42, **ds_kwargs):
    set_seed(seed)

    ds_params_str = ", ".join([f"{key}={val}" for key, val in ds_kwargs.items()])
    print("="*50)
    print(f"--- Beginning of {k}-fold Cross-Validation ---")
    print(f"Testing with: {ds_method} [{ds_params_str}] | Norm: {norm_opt} | LR: 0.001")
    print("="*50)

    # Initialize sklearn k-fold
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    cross_val_results = []
    best_epochs = []

    ssm_dir = Path('../data/Challenge-ABC/Train/SSM_Challenge-ABC')
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(train_ids), 1):
        print(f">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")
        print(f">>>>>>> Fold {fold}/{k} >>>>>>>")
        print(f">>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>")

        # map file ids to the splits
        fold_train_ids = [train_ids[i] for i in train_idx]
        fold_val_ids = [train_ids[i] for i in val_idx]

        fold_train_pts = [train_pts[i] for i in train_idx]
        fold_train_lb = [train_lb[i] for i in train_idx]
        fold_val_lb = [train_lb[i] for i in val_idx]

        print(f"Train models: {len(fold_train_ids)} | Val models: {len(fold_val_ids)}")

        # get the mean and std for the chosen train models to normalize with after
        X_chosen_train_list = []
        for f_id in fold_train_ids:
            ssm_file = ssm_dir / f"{f_id}.ssm"
            features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
            X_chosen_train_list.append(features)
        X_chosen_train_np = np.vstack(X_chosen_train_list)
        del X_chosen_train_list
        gc.collect()

        X_chosen_train_tensor = torch.from_numpy(X_chosen_train_np)
        _, fold_train_mean, fold_train_std = normalize_data_inplace(X_chosen_train_tensor, opt=norm_opt)

        del X_chosen_train_tensor, X_chosen_train_np
        torch.cuda.empty_cache()
        gc.collect()

        # perform the downsampling on the training fold 
        X_train_ds_list, y_train_ds_list = build_downsampled_features(
            points_list=fold_train_pts,
            labels_list=fold_train_lb,
            file_ids=fold_train_ids,
            method=ds_method,
            **ds_kwargs
        )
        X_train_raw = np.vstack(X_train_ds_list)
        y_train_raw = np.concatenate(y_train_ds_list)

        # load the ssm features for the validation fold
        X_val_list = []
        for f_id in fold_val_ids:
            ssm_file = ssm_dir / f"{f_id}.ssm"
            features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
            X_val_list.append(features)
        X_val_raw = np.vstack(X_val_list)

        del X_val_list 
        gc.collect()

        y_val_raw = np.concatenate(fold_val_lb)

        # transform to tensors
        X_train_tensor = torch.from_numpy(X_train_raw)
        y_train_tensor = torch.from_numpy(y_train_raw)
        X_val_tensor = torch.from_numpy(X_val_raw)
        y_val_tensor = torch.from_numpy(y_val_raw)

        # normalize based on the chosen training models
        X_train_tensor,_,_ = normalize_data_inplace(X_train_tensor,fold_train_mean, fold_train_std, opt=norm_opt)
        X_val_tensor,_,_ = normalize_data_inplace(X_val_tensor, fold_train_mean, fold_train_std, opt=norm_opt)
        
        # train the model
        history, _ = train_mlp_model(
            X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor,
            nb_epochs=50, batch_size=512, lr=0.001,
            device=device, display_metrics=False, patience=10
        )

        best_idx = history['best_epoch']-1
        fold_f1 = history['val_f1_1'][best_idx]
        fold_epoch = history['best_epoch']
        print(f"+++++++++++++++++++++++++++++++")
        print(f"+++++ Fold {fold} Results +++++")
        print(f"+++++++++++++++++++++++++++++++")
        print(f"Best Epoch: {fold_epoch} | F1-score (Edges): {fold_f1:.4f}")

        best_epochs.append(fold_epoch)
        cross_val_results.append({
            'fold': fold,
            'best_epoch': fold_epoch,
            'f1_class_1': fold_f1,
            'precision_1': history['val_precision_1'][best_idx],
            'recall_1': history['val_recall_1'][best_idx],
            'mcc': history['val_mcc'][best_idx],
        })

        # cleanup
        del X_train_raw, y_train_raw, X_val_raw, y_val_raw
        del X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor, history
        del X_train_ds_list, y_train_ds_list
        torch.cuda.empty_cache()
        gc.collect()
    
    df_cv = pd.DataFrame(cross_val_results)
    avg_epoch = int(np.round(np.mean(best_epochs)))
    avg_f1 = np.mean(df_cv['f1_class_1'])

    print("\n==================================================")
    print(f"--- {k}-Fold CV Completed ---")
    print(f"Target Training Duration : {avg_epoch} Epochs")
    print(f"Expected F1-Score        : {avg_f1:.4f}")
    print("==================================================")
    display(df_cv)
    
    return avg_epoch, df_cv

In [13]:
train_pts, train_lb, train_ids = load_full_dataset_ply_lb(dataset='Train')
kfold_cross_val_train(train_pts=train_pts, train_lb=train_lb, train_ids=train_ids, ds_method='voxel', resolution_percentage=0.02)

Total of 198 .ply and .lb files has been loaded successfully.
Global seed fixed on 42
--- Beginning of 5-fold Cross-Validation ---
Testing with: voxel [resolution_percentage=0.02] | Norm: flat | LR: 0.001
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
>>>>>>> Fold 1/5 >>>>>>>
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Train models: 158 | Val models: 40
Starting feature extraction using 'voxel' downsampling...
Feature extraction complete. Processed 158 valid files.

Global seed fixed on 42
Training on device: cuda (Seed: 42) | Features in input: 320
Start of training...

Early Stopping triggered at epoch 20. Returning best weigths to now.
Getting best model weights with val_loss = 0.0044
Training took 2.17 minutes.
+++++++++++++++++++++++++++++++
+++++ Fold 1 Results +++++
+++++++++++++++++++++++++++++++
Best Epoch: 10 | F1-score (Edges): 0.9897
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
>>>>>>> Fold 2/5 >>>>>>>
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
Train models: 158 | Val models: 40
Starting feature extraction using 'voxel' dow

,fold,best_epoch,f1_class_1,precision_1,recall_1,mcc
0,1,10,0.989695,0.990839,0.988554,0.989205
1,2,4,0.991054,0.993338,0.988782,0.990686
2,3,10,0.983330,0.975469,0.991318,0.982580
3,4,26,0.990347,0.992217,0.988484,0.989748
4,5,3,0.984783,0.986613,0.982960,0.984017


(11,
    fold  best_epoch  f1_class_1  precision_1  recall_1       mcc
 0     1          10    0.989695     0.990839  0.988554  0.989205
 1     2           4    0.991054     0.993338  0.988782  0.990686
 2     3          10    0.983330     0.975469  0.991318  0.982580
 3     4          26    0.990347     0.992217  0.988484  0.989748
 4     5           3    0.984783     0.986613  0.982960  0.984017)

In [13]:
def train_mlp_model_after_cv(X_train, y_train, nb_epochs=11, batch_size=512, lr=0.001, device='cuda', seed=42):
    set_seed(seed)
    input_dim = X_train.size(1)

    print(f"==================================================")
    print(f"Training with :")
    print(f"Epochs: {nb_epochs} | Batch: {batch_size} | LR: {lr} | X_train size : {X_train.size(0)}")
    print(f"Training on device: {device} (Seed: {seed}) | Features in input: {input_dim}")
    print(f"==================================================")
    
    start_time = time.time()

    model = MLP(input_dim=input_dim).to(device)
    X_train_gpu = X_train.to(device)
    y_train_gpu = y_train.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        'train_loss_epoch' : [], 'train_loss_step': []
    }

    model.train()
    for epoch in range(1, nb_epochs + 1):
        running_loss = 0.0
        idx = torch.randperm(X_train_gpu.size(0), device=device)
    
        for i in range(0, X_train_gpu.size(0), batch_size):
            batch_idx = idx[i : i + batch_size]
            inputs = X_train_gpu[batch_idx]
            labels = y_train_gpu[batch_idx]

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            history['train_loss_step'].append(loss.item())
            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / X_train_gpu.size(0)
        history['train_loss_epoch'].append(epoch_loss)
        print(f"Epoch [{epoch:02d}/{nb_epochs}] | Train Loss: {epoch_loss:.4f}")

    del X_train_gpu, y_train_gpu
    torch.cuda.empty_cache()

    end_time = time.time()
    print(f"Training took {(end_time - start_time) / 60:.2f} minutes.")
    
    return history, model

In [14]:
def get_train_mean_std(opt='flat', precompute_all=True):
    processed_dir=Path('../data/processed_data')
    processed_dir.mkdir(parents=True, exist_ok=True)

    stats_paths = {
        'flat': (processed_dir / 'global_train_mean_flat.pt', processed_dir / 'global_train_std_flat.pt'),
        'desc': (processed_dir / 'global_train_mean_desc.pt', processed_dir / 'global_train_std_desc.pt')
    }
    if opt not in stats_paths:
        raise ValueError(f"Unknown normalization option: '{opt}'. Must be one of {list(stats_paths.keys())}")
    
    req_mean_path, req_std_path = stats_paths[opt]

    # If what we need is already on disk return instantly
    if req_mean_path.exists() and req_std_path.exists():
        return torch.load(req_mean_path, weights_only=True), torch.load(req_std_path, weights_only=True)
    
    # else we must load the dataset 
    print(f"Stats for '{opt}' missing. Loading full dataset to compute normalization stats...")
    X_train_full_raw, _ = load_full_dataset_ssm_lb('Train') 
    X_train_full_tensor = torch.tensor(X_train_full_raw, dtype=torch.float32)
    del X_train_full_raw
    torch.cuda.empty_cache(); gc.collect()
    
    # we can compute missing stats for the other one too if wanted
    to_compute = stats_paths.items() if precompute_all else [(opt, stats_paths[opt])]
    
    for norm_opt, (mean_path, std_path) in to_compute:
        if not (mean_path.exists() and std_path.exists()):
            _, global_mean, global_std = normalize_data_inplace(X_train_full_tensor.clone(), opt=norm_opt)
            torch.save(global_mean, mean_path)
            torch.save(global_std, std_path)
       
    del X_train_full_tensor
    torch.cuda.empty_cache(); gc.collect()

    return torch.load(req_mean_path, weights_only=True), torch.load(req_std_path, weights_only=True)

In [ ]:
def test_mlp_cv_ds(target_epoch=11, ds_method='voxel', norm_opt='flat', device='cuda', seed=42, **ds_kwargs):
    set_seed(42)
    processed_dir = Path('../data/processed_data')
    processed_dir.mkdir(parents=True, exist_ok=True)

    # build the ds filename
    param_str = "_".join([f"{k}-{v}" for k, v in ds_kwargs.items()])
    ds_train_data_filename = processed_dir / f"ds_{ds_method}_{param_str}.pt"

    # disk check, if alr exists just load, otherwise compute and save
    if ds_train_data_filename.exists():
        print(f"Loading alr saved downsampled dataset: {ds_train_data_filename.name}")
        data = torch.load(ds_train_data_filename, weights_only=True)
        X_train_tensor= data['X_train'] 
        y_train_tensor = data['y_train']
        del data
        gc.collect()
    else:
        print(f"Generating downsampled dataset: {ds_train_data_filename.name}")
        # load and downsample the train dataset 
        train_pts, train_lb, train_ids = load_full_dataset_ply_lb(dataset='Train')

        X_train_ds_list, y_train_ds_list = build_downsampled_features(
            points_list=train_pts,
            labels_list=train_lb,
            file_ids=train_ids,
            method=ds_method,
            **ds_kwargs
        )
        X_train_ds_raw = np.vstack(X_train_ds_list)
        y_train_ds_raw = np.concatenate(y_train_ds_list)

        X_train_tensor = torch.tensor(X_train_ds_raw, dtype=torch.float)
        y_train_tensor = torch.tensor(y_train_ds_raw, dtype=torch.long)
        torch.save({'X_train': X_train_tensor, 'y_train': y_train_tensor}, ds_train_data_filename)
        del train_pts, train_lb, train_ids, X_train_ds_raw, y_train_ds_raw 
        torch.cuda.empty_cache(); gc.collect()


    # get mean+std and normalize data
    train_mean, train_std = get_train_mean_std(opt=norm_opt, precompute_all=False)
    X_train_tensor, _, _ = normalize_data_inplace(X_train_tensor, train_mean, train_std, opt=norm_opt)

    hist, model = train_mlp_model_after_cv(
        X_train=X_train_tensor,
        y_train=y_train_tensor,
        nb_epochs=target_epoch,
        batch_size=512,
        lr=0.001,
        device=device,
        seed=seed
    )

    # load the 50 Test models 
    print("\nLoading 50 Test models...")
    X_test_raw, y_test_raw = load_full_dataset_ssm_lb('Validation')
    X_test_tensor = torch.tensor(X_test_raw, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test_raw, dtype=torch.long)
    del X_test_raw, y_test_raw
    gc.collect()

    # apply the exact same global train anchor to the test data
    X_test_tensor, _, _ = normalize_data_inplace(X_test_tensor, train_mean, train_std, opt=norm_opt)

    # metrics
    metrics = {
        'test_loss': [], 'test_acc': [], 'test_mcc': [],
        'test_precision_0': [], 'test_precision_1': [], 
        'test_recall_0': [], 'test_recall_1': [], 
        'test_f1_0': [], 'test_f1_1': [], 
        'test_iou_0': [], 'test_iou_1': [],
        'test_tp':[], 'test_tn':[], 'test_fp':[], 'test_fn':[]
    }


    f1_metric = MulticlassF1Score(num_classes=2, average='none').to(device)
    precision_metric = MulticlassPrecision(num_classes=2, average='none').to(device)
    recall_metric = MulticlassRecall(num_classes=2, average='none').to(device)
    iou_metric = MulticlassJaccardIndex(num_classes=2, average='none').to(device)
    mcc_metric = BinaryMatthewsCorrCoef().to(device)
    acc_metric = BinaryAccuracy().to(device)
    conf_matrix_metric = MulticlassConfusionMatrix(num_classes=2).to(device)

    f1_metric.reset()
    precision_metric.reset()
    recall_metric.reset()
    mcc_metric.reset()
    iou_metric.reset()
    acc_metric.reset()
    conf_matrix_metric.reset()

    model.eval()
    X_test_gpu = X_test_tensor.to(device)
    y_test_gpu = y_test_tensor.to(device)
    running_test_loss = 0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        bs = 50000
        for i in range(0, X_test_gpu.size(0), bs):
            inputs = X_test_gpu[i : i + bs]
            labels = y_test_gpu[i : i + bs]

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_test_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)

            f1_metric.update(predicted, labels)
            precision_metric.update(predicted, labels)
            recall_metric.update(predicted, labels)
            mcc_metric.update(predicted, labels)
            iou_metric.update(predicted, labels)
            acc_metric.update(predicted, labels)
            conf_matrix_metric.update(predicted, labels)

        epoch_test_loss = running_test_loss / X_test_gpu.size(0)
        metrics['test_loss'].append(epoch_test_loss)
            
    # --- Compute Evaluation Metrics ---
    val_prec = precision_metric.compute()
    val_rec = recall_metric.compute()
    val_f1 = f1_metric.compute()
    val_iou = iou_metric.compute()
    conf_mat = conf_matrix_metric.compute()
        
    metrics['test_precision_0'].append(val_prec[0].item())
    metrics['test_precision_1'].append(val_prec[1].item())
    metrics['test_recall_0'].append(val_rec[0].item())
    metrics['test_recall_1'].append(val_rec[1].item())
    metrics['test_f1_0'].append(val_f1[0].item())
    metrics['test_f1_1'].append(val_f1[1].item())
    metrics['test_iou_0'].append(val_iou[0].item())
    metrics['test_iou_1'].append(val_iou[1].item())

    val_mcc = mcc_metric.compute().item()
    val_acc = acc_metric.compute().item() * 100.0
    metrics['test_mcc'].append(val_mcc)
    metrics['test_acc'].append(val_acc)

    metrics['test_tn'].append(conf_mat[0, 0].item())
    metrics['test_fp'].append(conf_mat[0, 1].item())
    metrics['test_fn'].append(conf_mat[1, 0].item())
    metrics['test_tp'].append(conf_mat[1, 1].item())

    print("\n==================================================")
    print("FINAL TEST SET RESULTS")
    print("==================================================")
    print(f"Test Loss     : {metrics['test_loss'][0]:.4f}")
    print(f"Test MCC      : {metrics['test_mcc'][0]:.4f}")
    print(f"Test Accuracy : {metrics['test_acc'][0]:.4f}\n")
    print(f"-- CLASS 1 --")
    print(f"F1-Score     : {metrics['test_f1_1'][0]:.4f}")
    print(f"Precision    : {metrics['test_precision_1'][0]:.4f}")
    print(f"Recall       : {metrics['test_recall_1'][0]:.4f}")
    print(f"IoU          : {metrics['test_iou_1'][0]:.4f}\n")
    print(f"-- CLASS 0 --")
    print(f"F1-Score     : {metrics['test_f1_0'][0]:.4f}")
    print(f"Precision    : {metrics['test_precision_0'][0]:.4f}")
    print(f"Recall       : {metrics['test_recall_0'][0]:.4f}")
    print(f"IoU          : {metrics['test_iou_0'][0]:.4f}\n")
    print(f"-- Confusion Matrix --")
    print(f"TP  : {metrics['test_tp'][0]:.4f}")
    print(f"FP  : {metrics['test_fp'][0]:.4f}")
    print(f"FN  : {metrics['test_fn'][0]:.4f}")
    print(f"TN  : {metrics['test_tn'][0]:.4f}")
    print("==================================================")  

In [ ]:
DOWNSAMPLING_GRID = {
    'voxel': [{'resolution_percentage': 0.01}, {'resolution_percentage': 0.02}, {'resolution_percentage': 0.03}, {'resolution_percentage': 0.04}, {'resolution_percentage': 0.05}],
    'fps': [{'retention_rate': 0.05}, {'retention_rate': 0.10}, {'retention_rate': 0.20}],
    'poisson': [{'radius': 1}, {'radius': 2}, {'radius': 3}],
    'random': [{'retention_rate': 0.05}, {'retention_rate': 0.10}, {'retention_rate': 0.20}],
    'uniform': [{'k_step': 5}, {'k_step': 10}, {'k_step': 20}]
}

LEARNING_RATES = [0.01, 0.005, 0.001]
BATCH_SIZES = [512, 1024]

Path('../data/processed_data').mkdir(parents=True, exist_ok=True)
Path('../data/results').mkdir(parents=True, exist_ok=True)

def generate_all_datasets(train_pts, train_lb, train_ids):
    processed_dir = Path('../data/processed_data')
    
    # Ensure directory exists
    if not processed_dir.exists(): 
        print(f"Error: datasets folder {processed_dir} not found.")
        return None

    print("\n--- Starting all downsamplings ---")
    for method, params_list in DOWNSAMPLING_GRID.items():
        for params in params_list:
            # create an identity filename (ex: ds_fps_retention_rate-0.1.pt)
            param_str = "_".join([f"{k}-{v}" for k, v in params.items()])
            filename = processed_dir / f"ds_{method}_{param_str}.pt"
            
            # safety check to not redo an already done treatment
            if filename.exists():
                print(f"✅ Skipping {filename.name} (Already computed)")
                continue
                
            print(f"⏳ Processing: {method} with {params} ...")
            set_seed(42)
            X_ds_list, y_ds_list = build_downsampled_features(
                points_list=train_pts, labels_list=train_lb, file_ids=train_ids, 
                method=method, **params
            )

            X_ds = np.vstack(X_ds_list)
            y_ds = np.concatenate(y_ds_list)
            del X_ds_list, y_ds_list
            torch.cuda.empty_cache(); gc.collect()

            X_tensor = torch.tensor(X_ds, dtype=torch.float32)
            y_tensor = torch.tensor(y_ds, dtype=torch.long)
            
            # save as a single dictionary inside a .pt file
            torch.save({'X_train': X_tensor, 'y_train': y_tensor}, filename)
            print(f"💾 Saved {filename.name} | Shape: {list(X_tensor.shape)}")
            
            # cleanup loop variables from the ram 
            del X_ds, y_ds, X_tensor, y_tensor
            torch.cuda.empty_cache(); gc.collect()

In [39]:
train_pts, train_lb, train_ids = load_full_dataset_ply_lb(dataset='Train')
generate_all_datasets(train_pts=train_pts, train_lb=train_lb, train_ids=train_ids)

Total of 198 .ply and .lb files has been loaded successfully.

--- Starting all downsamplings ---
⏳ Processing: voxel with {'resolution_percentage': 0.01} ...
Global seed fixed on 42
Starting feature extraction using 'voxel' downsampling...
Feature extraction complete. Processed 198 valid files.

💾 Saved ds_voxel_resolution_percentage-0.01.pt | Shape: [2550942, 20, 16]
⏳ Processing: voxel with {'resolution_percentage': 0.02} ...
Global seed fixed on 42
Starting feature extraction using 'voxel' downsampling...
Feature extraction complete. Processed 198 valid files.

💾 Saved ds_voxel_resolution_percentage-0.02.pt | Shape: [1113005, 20, 16]
⏳ Processing: voxel with {'resolution_percentage': 0.03} ...
Global seed fixed on 42
Starting feature extraction using 'voxel' downsampling...
Feature extraction complete. Processed 198 valid files.

💾 Saved ds_voxel_resolution_percentage-0.03.pt | Shape: [585047, 20, 16]
⏳ Processing: voxel with {'resolution_percentage': 0.04} ...
Global seed fixed on

In [ ]:
test_mlp_cv_ds(ds_method='voxel', resolution_percentage=0.02)

Global seed fixed on 42
Loading alr saved downsampled dataset: ds_voxel_resolution_percentage-0.02.pt
Stats for 'flat' missing. Loading full dataset to compute normalization stats...
Total of 198 .ssm and .lb files has been loaded successfully.
Global seed fixed on 42
Training with :
Epochs: 11 | Batch: 512 | LR: 0.001 | X_train size : 1113005
Training on device: cuda (Seed: 42) | Features in input: 320
Epoch [01/11] | Train Loss: 0.0154
Epoch [02/11] | Train Loss: 0.0053
Epoch [03/11] | Train Loss: 0.0035
Epoch [04/11] | Train Loss: 0.0021
Epoch [05/11] | Train Loss: 0.0021
Epoch [06/11] | Train Loss: 0.0019
Epoch [07/11] | Train Loss: 0.0013
Epoch [08/11] | Train Loss: 0.0012
Epoch [09/11] | Train Loss: 0.0011
Epoch [10/11] | Train Loss: 0.0009
Epoch [11/11] | Train Loss: 0.0009
Training took 1.19 minutes.

Loading 50 Test models...
Total of 50 .ssm and .lb files has been loaded successfully.

FINAL TEST SET RESULTS
Test Loss     : 0.0099
Test MCC      : 0.9865
Test Accuracy : 99.851

In [ ]:
def run_grid_search(device='cuda', val_dataset_name='Validation'):
    processed_dir = Path('../data/processed_data')
    results_file = Path('../data/results/grid_search_metrics_v1.csv')
    
    # load global train stats 
    print("--- Loading Global Training Statistics ---")
    mean_flat, std_flat = get_train_mean_std(opt='flat', precompute_all=True)
    mean_desc, std_desc = get_train_mean_std(opt='desc')
    global_stats = {
        'flat': {'mean': mean_flat, 'std': std_flat},
        'desc': {'mean': mean_desc, 'std': std_desc}
    }

    # load validation dataset once, here I'm intentionally using the 50 models to validate so we can get an idea on how well a model can perform with a chosen method 
    print("--- Loading Full Validation Dataset into RAM ---")
    X_val_raw, y_val_raw = load_full_dataset_ssm_lb(val_dataset_name)
    X_val_base = torch.tensor(X_val_raw, dtype=torch.float32)
    y_val_base = torch.tensor(y_val_raw, dtype=torch.long)
    del X_val_raw, y_val_raw
    gc.collect()

    all_results = []
    completed_runs = set()

    # safety check for resuming crashes
    if results_file.exists():
        print(f"Found existing results file at {results_file}. Parsing completed runs...")
        df_existing = pd.read_csv(results_file)
        
        for _, row in df_existing.iterrows():
            s = f"{row['method']}_{row['ds_param']}_{row['norm_opt']}_{row['learning_rate']}_{row['batch_size']}"
            completed_runs.add(s)
            
        all_results = df_existing.to_dict('records')
        print(f"Resuming experiments. {len(completed_runs)} runs already completed.")
    else:
        results_file.parent.mkdir(parents=True, exist_ok=True)
        print("No existing results found. Starting fresh.")

    dataset_files = sorted(list(processed_dir.glob('ds_*.pt')))
    NORM_OPTS = ['flat', 'desc']
    
    total_runs = len(dataset_files) * len(NORM_OPTS) * len(LEARNING_RATES) * len(BATCH_SIZES)
    current_run = 0

    for ds_file in dataset_files:
        file_parts = ds_file.stem.split('_')
        method = file_parts[1]
        param_str = "_".join(file_parts[2:])
        
        # load the native PyTorch dictionary
        data = torch.load(ds_file, weights_only=True)
        X_train_base = data['X_train']
        y_train_base = data['y_train']
        train_pts_nb = X_train_base.size(0)

        for norm_opt in NORM_OPTS:
            # clone outside LR and BS loops
            X_train_norm = X_train_base.clone()
            X_val_norm = X_val_base.clone()
            
            # qpply global stats 
            g_mean = global_stats[norm_opt]['mean']
            g_std = global_stats[norm_opt]['std']
            
            X_train_norm, _, _ = normalize_data_inplace(X_train_norm, mean=g_mean, std=g_std, opt=norm_opt)
            X_val_norm, _, _ = normalize_data_inplace(X_val_norm, mean=g_mean, std=g_std, opt=norm_opt)
            
            for lr in LEARNING_RATES:
                for bs in BATCH_SIZES:
                    current_run += 1
                    current_s = f"{method}_{param_str}_{norm_opt}_{lr}_{bs}"
                    
                    if current_s in completed_runs:
                        print(f"[{current_run}/{total_runs}] ⏭️ Skipping: {method} | {param_str} | Norm: {norm_opt} | LR: {lr} | BS: {bs}")
                        continue
                        
                    print(f"\n[{current_run}/{total_runs}] 🚀 Running: {method} | {param_str} | Norm: {norm_opt} | LR: {lr} | BS: {bs}")
                    
                    history, _ = train_mlp_model(
                        X_train_norm, y_train_base, X_val_norm, y_val_base,
                        nb_epochs=50, batch_size=bs, lr=lr, 
                        device=device, display_metrics=False, patience=10,
                    )
                    
                    best_idx = history['best_epoch'] - 1
                    
                    run_record = {
                        'method': method,
                        'ds_param': param_str,
                        'train_pts_nb': train_pts_nb,
                        'norm_opt': norm_opt,
                        'learning_rate': lr,
                        'batch_size': bs,
                        'best_epoch': history['best_epoch'],
                        'val_loss': history['val_loss'][best_idx],
                        'mcc': history['val_mcc'][best_idx],
                        'f1_class_1': history['val_f1_1'][best_idx],
                        'f1_class_0': history['val_f1_0'][best_idx],
                        'precision_1': history['val_precision_1'][best_idx],
                        'precision_0': history['val_precision_0'][best_idx],
                        'recall_1': history['val_recall_1'][best_idx],
                        'recall_0': history['val_recall_0'][best_idx],
                        'iou_1': history['val_iou_1'][best_idx],
                        'iou_0': history['val_iou_0'][best_idx],
                        'fp': history['val_fp'][best_idx],
                        'fn': history['val_fn'][best_idx],
                        'tp': history['val_tp'][best_idx],
                        'tn': history['val_tn'][best_idx],
                    }
                    
                    all_results.append(run_record)
                    completed_runs.add(current_s)
                    
                    # overwrite CSV every time we add a row so if it crashes we would still have the history
                    df = pd.DataFrame(all_results)
                    df.to_csv(results_file, index=False)
                    
                    del history
                    torch.cuda.empty_cache()
            
            # clean up normalized tensors before switching to the next norm_opt
            del X_train_norm, X_val_norm
            torch.cuda.empty_cache(); gc.collect()
                        
        # Free the loaded downsampled dataset from memory
        del X_train_base, y_train_base, data
        torch.cuda.empty_cache(); gc.collect()

In [24]:
run_grid_search()

--- Loading Global Training Statistics ---
--- Loading Full Validation Dataset into RAM ---
Total of 50 .ssm and .lb files has been loaded successfully.
Found existing results file at ../data/results/grid_search_metrics_v1.csv. Parsing completed runs...
Resuming experiments. 204 runs already completed.
[1/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: flat | LR: 0.01 | BS: 512
[2/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: flat | LR: 0.01 | BS: 1024
[3/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: flat | LR: 0.005 | BS: 512
[4/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: flat | LR: 0.005 | BS: 1024
[5/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: flat | LR: 0.001 | BS: 512
[6/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: flat | LR: 0.001 | BS: 1024
[7/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: desc | LR: 0.01 | BS: 512
[8/204] ⏭️ Skipping: fps | retention_rate-0.05 | Norm: desc | LR: 0.01 | BS: 1024
[9/204] ⏭️ Skipping: fps | retention_rat